<a href="https://colab.research.google.com/github/samcordner/interperability-notes-monorepo/blob/main/projects/find_induction_heads.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformer_lens -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.6 MB/s eta 0:00:00


In [3]:
import torch
import einops
from transformer_lens import HookedTransformer
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "colab"

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)

print(f"Loaded {model.cfg.model_name}")
print(f"Layers: {model.cfg.n_layers}, Heads per layer: {model.cfg.n_heads}")

/tmp/ipykernel_3003/616067636.py:2: DeprecationWarning:

HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.



config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loaded pretrained model gpt2 into HookedTransformer
Loaded gpt2
Layers: 12, Heads per layer: 12


In [5]:
def generate_repeated_tokens(model, seq_len=25, batch=1, seed=0):
    torch.manual_seed(seed)
    prefix = torch.full((batch, 1), model.tokenizer.bos_token_id, dtype=torch.long)
    rand_tokens = torch.randint(1000, model.cfg.d_vocab, (batch, seq_len))
    rep_tokens = torch.cat([prefix, rand_tokens, rand_tokens], dim=1).to(device)
    return rep_tokens

seq_len = 25
batch = 1
rep_tokens = generate_repeated_tokens(model, seq_len=seq_len, batch=batch)

print("Sequence shape:", rep_tokens.shape)
print(model.to_str_tokens(rep_tokens[0]))

Sequence shape: torch.Size([1, 51])
['<|endoftext|>', 'alos', ' perfectly', ' nihil', ' cryptographic', ' 383', ' politic', 'tle', ' Atlanta', ' Ibrahim', ' Houses', ' Poc', 'atal', 'MAX', ' Doesn', 'chi', ' Bush', 'limit', '————————', ' Hu', 'icles', 'oslav', ' remained', ' vol', ' Ninth', ' catalogue', 'alos', ' perfectly', ' nihil', ' cryptographic', ' 383', ' politic', 'tle', ' Atlanta', ' Ibrahim', ' Houses', ' Poc', 'atal', 'MAX', ' Doesn', 'chi', ' Bush', 'limit', '————————', ' Hu', 'icles', 'oslav', ' remained', ' vol', ' Ninth', ' catalogue']


In [6]:
rep_logits, cache = model.run_with_cache(rep_tokens)

layer_to_inspect = 5
attn_patterns = cache["pattern", layer_to_inspect]
print("Attention pattern shape:", attn_patterns.shape)

Attention pattern shape: torch.Size([1, 12, 51, 51])


In [7]:
def plot_attention_pattern(cache, layer, head, tokens, model):
    pattern = cache["pattern", layer][0, head]
    str_tokens = model.to_str_tokens(tokens[0])
    fig = px.imshow(
        pattern.cpu().numpy(),
        labels=dict(x="Key position", y="Query position", color="Attention"),
        x=str_tokens,
        y=str_tokens,
        title=f"Layer {layer}, Head {head} attention pattern",
        color_continuous_scale="Blues",
    )
    fig.update_layout(width=800, height=800)
    fig.show()

In [8]:
def induction_score(model, seq_len=25, batch=8, n_seeds=5):
    n_layers, n_heads = model.cfg.n_layers, model.cfg.n_heads
    scores = torch.zeros(n_layers, n_heads, device=device)

    for seed in range(n_seeds):
        rep_tokens = generate_repeated_tokens(model, seq_len=seq_len, batch=batch, seed=seed)
        _, cache = model.run_with_cache(
            rep_tokens,
            names_filter=lambda name: name.endswith("pattern"),
        )
        for layer in range(n_layers):
            pattern = cache["pattern", layer]
            induction_stripe = pattern.diagonal(offset=-(seq_len - 1), dim1=-2, dim2=-1)
            scores[layer] += induction_stripe.mean(dim=(0, -1))

    scores /= n_seeds
    return scores

scores = induction_score(model, seq_len=seq_len, batch=8, n_seeds=5)

fig = px.imshow(
    scores.cpu().numpy(),
    labels=dict(x="Head", y="Layer", color="Induction score"),
    title="Induction score per head (GPT-2 small)",
    color_continuous_scale="Reds",
)
fig.update_layout(width=600, height=600)
fig.show()

flat_scores = scores.flatten()
top_vals, top_idxs = flat_scores.topk(5)
print("Top induction heads (layer, head, score):")
for val, idx in zip(top_vals, top_idxs):
    layer = idx.item() // model.cfg.n_heads
    head = idx.item() % model.cfg.n_heads
    print(f"  Layer {layer}, Head {head}: {val.item():.3f}")

Top induction heads (layer, head, score):
  Layer 5, Head 5: 0.893
  Layer 5, Head 1: 0.860
  Layer 6, Head 9: 0.857
  Layer 7, Head 10: 0.856
  Layer 7, Head 2: 0.768


In [9]:
# fill in whichever layer/head came out on top from Cell 7
top_layer, top_head = 5, 5

_, cache = model.run_with_cache(rep_tokens)
plot_attention_pattern(cache, top_layer, top_head, rep_tokens, model)

In [10]:
pattern = cache["pattern", top_layer][0, top_head]
str_tokens = model.to_str_tokens(rep_tokens[0])

# Only show second-copy queries (rows) vs full sequence (columns)
start = seq_len + 1  # where the second copy begins
zoomed_pattern = pattern[start:, :seq_len+1]  # second-copy queries, first-copy keys
zoomed_y_labels = str_tokens[start:]
zoomed_x_labels = str_tokens[:seq_len+1]

fig = px.imshow(
    zoomed_pattern.cpu().numpy(),
    labels=dict(x="Key position (first copy)", y="Query position (second copy)", color="Attention"),
    x=zoomed_x_labels,
    y=zoomed_y_labels,
    title=f"Layer {top_layer}, Head {top_head} — zoomed to induction region",
    color_continuous_scale="Blues",
)
fig.update_layout(width=700, height=700)
fig.show()